# Registro de errores — Reporte Excel y correo

## Propósito

Registrar en `etl_log` el error de la task `gld_reporte_excel_correo` y avisar por Slack. Se ejecuta solo cuando la task del reporte falla.

## Flujo

1. Recuperar el `log_id` publicado por `gld_reporte_excel_correo`.
2. Consultar el detalle de la excepción mediante Databricks Jobs API.
3. Actualizar el registro en `etl_log` con estado `ERROR`, fecha de fin y detalle.
4. Enviar la alerta a Slack.

## Configuración en el Workflow

Task `log_error_gld_reporte_excel_correo`, dependiente solo de `gld_reporte_excel_correo`, con condición de ejecución **At least one failed**. Ninguna otra task debe depender de ella.

In [ ]:
# ============================================================
# OBTENCIÓN DEL LOG_ID Y PARÁMETROS DEL TASK FALLIDO
# ============================================================

dbutils.widgets.text("error_code", "UNKNOWN_ERROR")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("result_state", "FAILED")

error_code = dbutils.widgets.get("error_code")
run_id = dbutils.widgets.get("run_id")
result_state = dbutils.widgets.get("result_state")

log_id = dbutils.jobs.taskValues.get(
    taskKey="gld_reporte_excel_correo",
    key="log_id",
    debugValue=""
)

print(f"✓ log_id: {log_id}")
print(f"✓ error_code: {error_code}")
print(f"✓ run_id: {run_id}")
print(f"✓ result_state: {result_state}")

In [ ]:
# ============================================================
# OBTENCIÓN DEL DETALLE DEL ERROR MEDIANTE JOBS API
# ============================================================

import requests

error_detail = None

if run_id and str(run_id).isdigit() and str(run_id) != "0":
    try:
        workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
        token = (
            dbutils.notebook.entry_point
            .getDbutils()
            .notebook()
            .getContext()
            .apiToken()
            .get()
        )

        url = f"https://{workspace_url}/api/2.2/jobs/runs/get-output"

        response = requests.get(
            url,
            headers={"Authorization": f"Bearer {token}"},
            params={"run_id": str(run_id)},
            timeout=30
        )

        if response.status_code == 200:
            output_info = response.json()
            error_detail = (
                output_info.get("error")
                or output_info.get("error_trace")
                or (output_info.get("notebook_output") or {}).get("result")
            )
        else:
            error_detail = f"Error al consultar la API (Status {response.status_code}): {response.text}"

    except Exception as ex:
        error_detail = f"Excepción durante la conexión a Jobs API: {str(ex)}"

if not error_detail:
    error_detail = "No fue posible obtener la traza del error mediante Jobs API (Run ID no válido o sin respuesta)."

print("✓ Detalle del error obtenido:")
print(error_detail)

In [ ]:
# ============================================================
# CONSTRUCCIÓN DEL MENSAJE DE ERROR
# ============================================================

mensaje_error = (
    f"Task: gld_reporte_excel_correo | "
    f"Error code: {error_code} | "
    f"Run ID: {run_id} | "
    f"Resultado: {result_state} | "
    f"Detalle: {error_detail}"
)

print("Mensaje de error:")
print(mensaje_error)

In [ ]:
# ============================================================
# ACTUALIZACIÓN DEL LOG COMO ERROR
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

delta_log = DeltaTable.forName(spark, "prod_dataco.metadata.etl_log")

if log_id:
    delta_log.update(
        condition=F.col("log_id") == log_id,
        set={
            "fecha_fin": F.current_timestamp(),
            "estado": F.lit("ERROR"),
            "mensaje_error": F.lit(mensaje_error[:2000])
        }
    )
    print(f"✓ etl_log actualizado como ERROR: {log_id}")
else:
    print("⚠️ No se realizó la actualización: log_id no fue recibido.")

In [ ]:
# ============================================================
# ALERTA A SLACK
# ============================================================

import requests as slack_requests

try:
    webhook_url = dbutils.secrets.get(scope="alerts", key="slack_webhook_url")

    slack_payload = {
        "text": (
            f":rotating_light: *Fallo en pipeline DataCo*\n"
            f"*Task:* gld_reporte_excel_correo\n"
            f"*Estado:* {result_state}\n"
            f"*Error code:* {error_code}\n"
            f"*Run ID:* {run_id}\n"
            f"*log_id:* {log_id}"
        )
    }

    slack_response = slack_requests.post(webhook_url, json=slack_payload, timeout=10)

    if slack_response.status_code == 200:
        print("✓ Alerta enviada a Slack")
    else:
        print(f"⚠️ Slack respondió con status {slack_response.status_code}")

except Exception as ex:
    print(f"⚠️ No se pudo enviar la alerta a Slack: {ex}")